# LightGBM Regressor: Log Return Prediction
This notebook trains a LightGBM model on log returns and evaluates performance in price space.

In [ ]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from pathlib import Path

# Handle paths based on execution directory
if Path.cwd().name == 'notebooks':
    base_dir = Path('..')
else:
    base_dir = Path('.')

data_dir = base_dir / 'data' / 'processed'
assets_dir = base_dir / 'assets'
assets_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(data_dir / 'train.csv')
test_df = pd.read_csv(data_dir / 'test.csv')

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

features = ['lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'city_enc']
target = 'log_return'

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, num_leaves=31, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred_log = model.predict(X_test_scaled)

# Evaluation in Price Space
test_df['pred_log_return'] = y_pred_log

# Create price_lag_1 for back-transformation
test_df = test_df.sort_values(['RegionID', 'date'])
test_df['price_lag_1'] = test_df.groupby('RegionID')['price'].shift(1)
test_df['price_lag_1'] = test_df['price_lag_1'].ffill()

test_df['pred_price'] = test_df['price_lag_1'] * np.exp(test_df['pred_log_return'])

mae_p = mean_absolute_error(test_df['price'], test_df['pred_price'])
r2_p = r2_score(test_df['price'], test_df['pred_price'])

print(f"\n[Log Return Model] Price Space Metrics:")
print(f"MAE: ${mae_p:.2f}")
print(f"R2: {r2_p:.6f}")

# Visualization
daily_results = test_df.groupby('date')[['price', 'pred_price']].mean().reset_index()

plt.figure(figsize=(14, 7))
plt.plot(daily_results['date'], daily_results['price'], label='Average Actual Price', color='#2ecc71', linewidth=2, marker='o', markersize=4)
plt.plot(daily_results['date'], daily_results['pred_price'], label='Average Predicted Price', color='#3498db', linestyle='--', linewidth=2)
plt.title('House Price Trends: Actual vs Predicted (Log Return model)')
plt.xlabel('Date')
plt.ylabel('Average Price ($)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(assets_dir / 'price_predictions_date.png', dpi=300)
plt.show()

joblib.dump(model, assets_dir / 'housing_model.pkl')
joblib.dump(scaler, assets_dir / 'scaler.pkl')